# Experimento 2: Extraccion de Estado Inicial

**TFG: VisionLeaderUFV**  
**Author:** Alberto Gomez Soteres  
**Universidad Francisco de Vitoria**

---

## Objetivo

Evaluar la capacidad de los modelos PaliGemma para extraer el estado inicial del sistema:

- **Building** (EPS, CD, H) - cuando es visible
- **Floor** (-1, 0, 1, 2) - siempre visible

## Metodologia

1. Cargar modelo PaliGemma
2. Cargar dataset de carteles (building_sign/)
3. Ejecutar pregunta unica por imagen
4. Calcular metricas: Building Accuracy, Floor Accuracy, Location Accuracy
5. Seleccionar mejor modelo

## Metricas

- **Building Accuracy**: % correcto cuando Building es visible
- **Floor Accuracy**: % correcto en todas las imagenes
- **Location Accuracy**: % correcto en Building AND Floor (metrica principal)

## 1. Instalacion de dependencias

In [ ]:
!pip install -q transformers>=4.40.0 accelerate torch torchvision pillow huggingface_hub

## 2. Login en Hugging Face

In [ ]:
from huggingface_hub import login

# Login interactivo - pega tu token de HuggingFace
login()

## 3. Seleccion del modelo

In [ ]:
# ============================================================
# CONFIGURACION DEL MODELO
# ============================================================

# Modelos disponibles (descomenta el que quieras probar):

# PaliGemma 1 (3B params)
MODEL_ID = "google/paligemma-3b-mix-448"
# MODEL_ID = "google/paligemma-3b-mix-224"
# MODEL_ID = "google/paligemma-3b-pt-448"
# MODEL_ID = "google/paligemma-3b-pt-224"

# PaliGemma 2 (3B params)
# MODEL_ID = "google/paligemma2-3b-mix-448"
# MODEL_ID = "google/paligemma2-3b-mix-224"
# MODEL_ID = "google/paligemma2-3b-pt-448"
# MODEL_ID = "google/paligemma2-3b-pt-224"

# PaliGemma 2 (10B params) - Requiere mas VRAM
# MODEL_ID = "google/paligemma2-10b-mix-448"
# MODEL_ID = "google/paligemma2-10b-pt-448"

print(f"Modelo seleccionado: {MODEL_ID}")

## 4. Cargar modelo

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# Configuracion de dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print(f"Dispositivo: {device}")
print(f"Cargando modelo: {MODEL_ID}...")

# Cargar processor y modelo
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto"
).eval()

print("Modelo cargado correctamente!")

# Mostrar uso de VRAM
if torch.cuda.is_available():
    vram_gb = torch.cuda.memory_allocated() / (1024**3)
    print(f"VRAM usada: {vram_gb:.2f} GB")

## 5. Funcion de inferencia

In [ ]:
from PIL import Image
import time

def ask_paligemma(image: Image.Image, question: str, max_tokens: int = 100) -> dict:
    """
    Hace una pregunta a PaliGemma sobre una imagen.
    
    Args:
        image: Imagen PIL
        question: Pregunta en texto
        max_tokens: Maximo de tokens a generar
    
    Returns:
        Dict con respuesta y tiempo de inferencia
    """
    # Preparar inputs
    inputs = processor(text=question, images=image, return_tensors="pt").to(device, dtype=dtype)
    input_len = inputs["input_ids"].shape[-1]
    
    # Inferencia
    start_time = time.time()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False
        )
    inference_time = time.time() - start_time
    
    # Decodificar respuesta
    answer = processor.decode(output[0][input_len:], skip_special_tokens=True)
    
    return {
        "answer": answer.strip(),
        "inference_time": inference_time
    }

print("Funcion de inferencia lista!")

## 6. Configuracion del experimento

In [ ]:
# ============================================================
# PROMPT OPTIMIZADO PARA EXTRACCION DE ESTADO INICIAL
# ============================================================

QUESTION = """Look at this campus building sign carefully.

The building code (EPS, CD, or H) is displayed prominently, usually as large letters.
The floor number (-1, 0, 1, or 2) is typically shown in the top-right corner of the sign, often inside a small box or circle.

Extract ONLY:
1. Building code (EPS, CD, or H)
2. Floor number (-1, 0, 1, or 2)

Answer in this EXACT format:
Building: [CODE], Floor: [NUMBER]

Examples:
- Building: EPS, Floor: 0
- Building: CD, Floor: -1
- Building: H, Floor: 2

If the building code is not visible, write:
Building: none, Floor: [NUMBER]

Answer ONLY with the format above, nothing else."""

print("PROMPT OPTIMIZADO:")
print("="*60)
print(QUESTION)
print("="*60)

## 7. Probar con una imagen

In [ ]:
# ============================================================
# SUBIR Y PROBAR CON UNA IMAGEN
# ============================================================

from google.colab import files
import io

# Subir imagen
print("Sube una imagen de cartel del campus:")
uploaded = files.upload()

# Cargar imagen
filename = list(uploaded.keys())[0]
image = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")

print(f"\nImagen cargada: {filename}")
print(f"Tamano original: {image.size}")

# Mostrar imagen
display(image.resize((500, 400)))

# Ejecutar pregunta
print("\n" + "="*60)
print("EJECUTANDO PREGUNTA A PALIGEMMA")
print("="*60)
print(f"Modelo: {MODEL_ID}")
print("="*60)

result = ask_paligemma(image, QUESTION)

print("\n" + "="*60)
print("RESPUESTA DE PALIGEMMA:")
print("="*60)
print(f"\n>>> {result['answer']}\n")
print("="*60)
print(f"Tiempo de inferencia: {result['inference_time']:.2f} segundos")
print("="*60)

## 8. Probar con otra imagen (opcional)

In [ ]:
# ============================================================
# PROBAR CON OTRA IMAGEN
# ============================================================
# Ejecuta esta celda para probar con otra imagen sin recargar el modelo

from google.colab import files
import io

print("Sube otra imagen:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
image = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")

print(f"\nImagen: {filename}")
display(image.resize((500, 400)))

# Ejecutar
result = ask_paligemma(image, QUESTION)

print("\n" + "="*60)
print("RESPUESTA DE PALIGEMMA:")
print("="*60)
print(f"\n>>> {result['answer']}\n")
print(f"Tiempo: {result['inference_time']:.2f}s")
print("="*60)

## 9. Probar con prompts alternativos (opcional)

In [ ]:
# ============================================================
# PROMPT ALTERNATIVO - MAS CORTO
# ============================================================

PROMPT_SHORT = """Read the campus sign. The building code (EPS/CD/H) is shown as large letters. The floor number (-1/0/1/2) is in the top-right corner.

Answer ONLY as: Building: [CODE], Floor: [NUMBER]"""

print("PROMPT CORTO:")
print(PROMPT_SHORT)
print("\n" + "="*60)

# Ejecutar con prompt corto
result_short = ask_paligemma(image, PROMPT_SHORT)

print("RESPUESTA:")
print(f">>> {result_short['answer']}")
print(f"Tiempo: {result_short['inference_time']:.2f}s")

In [ ]:
# ============================================================
# PROMPT ALTERNATIVO - MUY DIRECTO
# ============================================================

PROMPT_DIRECT = """What building code and floor number are shown on this sign?
Answer as: Building: [CODE], Floor: [NUMBER]"""

print("PROMPT DIRECTO:")
print(PROMPT_DIRECT)
print("\n" + "="*60)

result_direct = ask_paligemma(image, PROMPT_DIRECT)

print("RESPUESTA:")
print(f">>> {result_direct['answer']}")
print(f"Tiempo: {result_direct['inference_time']:.2f}s")

## 10. Comparar resultados de los 3 prompts

In [ ]:
# ============================================================
# COMPARAR LOS 3 PROMPTS
# ============================================================

print("="*60)
print("COMPARACION DE PROMPTS")
print("="*60)
print(f"\nImagen: {filename}")
print("\n" + "-"*60)

print("\n[1. PROMPT OPTIMIZADO (largo)]")
print(f"Respuesta: {result['answer']}")
print(f"Tiempo: {result['inference_time']:.2f}s")

print("\n[2. PROMPT CORTO]")
print(f"Respuesta: {result_short['answer']}")
print(f"Tiempo: {result_short['inference_time']:.2f}s")

print("\n[3. PROMPT DIRECTO]")
print(f"Respuesta: {result_direct['answer']}")
print(f"Tiempo: {result_direct['inference_time']:.2f}s")

print("\n" + "="*60)
print("CONCLUSION: Elige el prompt que mejor funcione con tus imagenes")
print("="*60)

## 11. Cargar dataset y expected answers (para evaluacion completa)

In [ ]:
import json
from pathlib import Path
import os

# ============================================================
# CONFIGURACION DE RUTAS
# ============================================================

# Ajusta estas rutas segun tu estructura
DATA_PATH = "data/UFV Campus/building_sign"
EXPECTED_ANSWERS_PATH = "config/ufv_building_sign_expected_answers.json"

# Cargar expected answers
if os.path.exists(EXPECTED_ANSWERS_PATH):
    with open(EXPECTED_ANSWERS_PATH, 'r', encoding='utf-8') as f:
        EXPECTED_ANSWERS = json.load(f)
    print(f"Expected answers cargadas: {len(EXPECTED_ANSWERS)} imagenes")
else:
    print(f"Archivo no encontrado: {EXPECTED_ANSWERS_PATH}")
    print("Crea el archivo con las expected answers primero")
    EXPECTED_ANSWERS = {}

# Cargar imagenes del dataset
image_files = []
if os.path.exists(DATA_PATH):
    for img_file in Path(DATA_PATH).glob("*.JPEG"):
        if img_file.name in EXPECTED_ANSWERS:
            image_files.append(img_file)
    print(f"Imagenes encontradas: {len(image_files)}")
else:
    print(f"Carpeta no encontrada: {DATA_PATH}")
    print("Asegurate de tener las imagenes en la carpeta correcta")

print(f"\nTotal imagenes para evaluar: {len(image_files)}")

## 12. Ejecutar evaluacion

In [ ]:
# ============================================================
# EJECUTAR EVALUACION
# ============================================================

print("="*70)
print("EJECUTANDO EVALUACION")
print("="*70)
print(f"Modelo: {MODEL_ID}")
print(f"Total imagenes: {len(image_files)}")
print("="*70)

results = []

for img_file in image_files:
    img_name = img_file.name
    
    # Cargar imagen
    image = Image.open(img_file).convert("RGB")
    
    # Ejecutar pregunta
    result = ask_paligemma(image, QUESTION)
    
    # Guardar resultado
    results.append({
        "image": img_name,
        "prediction": result["answer"],
        "expected": EXPECTED_ANSWERS.get(img_name, {}),
        "inference_time": result["inference_time"]
    })
    
    print(f"\n[{len(results)}/{len(image_files)}] {img_name}")
    print(f"  Prediccion: {result['answer']}")
    print(f"  Tiempo: {result['inference_time']:.2f}s")

print("\n" + "="*70)
print("EVALUACION COMPLETADA")
print("="*70)

## 13. Funciones de extraccion

In [ ]:
import re

# ============================================================
# FUNCIONES DE EXTRACCION
# ============================================================

def extract_building_code(text: str) -> str:
    """Extrae codigo de edificio de una respuesta."""
    text_upper = text.upper().strip()
    
    # Buscar codigos conocidos
    known_codes = ["EPS", "CD", "H", "B"]
    for code in known_codes:
        if code in text_upper:
            return code
    
    # Detectar formato "EPS 0" o "CD -1"
    simple_pattern = r"^(EPS|CD|H|B)\s+-?\d+"
    match = re.match(simple_pattern, text_upper)
    if match:
        return match.group(1)
    
    # Detectar formato estructurado "Building: EPS"
    structured_pattern = r"building\s*[:-]?\s*(EPS|CD|H|B)"
    match = re.search(structured_pattern, text_upper)
    if match:
        return match.group(1)
    
    return None


def extract_floor_number(text: str) -> str:
    """Extrae numero de planta de una respuesta."""
    text_clean = text.strip()
    
    # Patrones para plantas (formato estructurado)
    patterns = [
        r"floor\s*[:-]?\s*(-?\d+)",
        r"planta\s*[:-]?\s*(-?\d+)",
        r"level\s*[:-]?\s*(-?\d+)",
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text_clean.lower())
        if match:
            return match.group(1)
    
    # Detectar formato simple "EPS 0" o "CD -1"
    simple_pattern = r"(?:EPS|CD|H|B|BUILDING|EDIFICIO)\s+(-?\d+)\b"
    match = re.search(simple_pattern, text_clean.upper())
    if match:
        return match.group(1)
    
    # Detectar numero suelto al final "0" o "-1"
    end_number_pattern = r"\s+(-?\d+)\s*$"
    match = re.search(end_number_pattern, text_clean)
    if match:
        return match.group(1)
    
    # Si la respuesta es solo un numero
    if re.match(r"^-?\d+$", text_clean):
        return text_clean
    
    return None


def parse_simple_response(response: str) -> dict:
    """Parsea respuestas simples como 'EPS 0'."""
    response = response.strip()
    result = {"building": None, "floor": None}
    
    # Formato simple "EPS 0"
    simple_pattern = r"^([A-Z]+)\s+(-?\d+)$"
    match = re.match(simple_pattern, response.upper())
    if match:
        result["building"] = match.group(1)
        result["floor"] = match.group(2)
        return result
    
    # Solo numero "0"
    if re.match(r"^-?\d+$", response):
        result["floor"] = response
        return result
    
    return None


print("Funciones de extraccion listas!")

In [ ]:
# ============================================================
# CALCULAR METRICAS
# ============================================================

# Contadores
building_correct = 0
building_total = 0
floor_correct = 0
floor_total = 0
location_correct = 0
location_total = 0

# Detalles por imagen
detailed_results = []

for result in results:
    img_name = result["image"]
    prediction = result["prediction"]
    expected = result["expected"]
    
    # Extraer valores esperados
    exp_building = expected.get("building")
    exp_floor = expected.get("floor")
    
    # Intentar parsear respuesta simple primero
    simple_parsed = parse_simple_response(prediction)
    if simple_parsed:
        pred_building = simple_parsed.get("building")
        pred_floor = simple_parsed.get("floor")
    else:
        pred_building = extract_building_code(prediction)
        pred_floor = extract_floor_number(prediction)
    
    # Evaluar Building
    building_ok = None
    if exp_building is not None:
        building_total += 1
        building_ok = (pred_building and pred_building.upper() == exp_building.upper())
        if building_ok:
            building_correct += 1
    
    # Evaluar Floor
    floor_total += 1
    floor_ok = (pred_floor and pred_floor == exp_floor)
    if floor_ok:
        floor_correct += 1
    
    # Evaluar Location (Building AND Floor)
    location_ok = None
    if exp_building is not None:
        location_total += 1
        location_ok = building_ok and floor_ok
        if location_ok:
            location_correct += 1
    
    # Guardar detalle
    detailed_results.append({
        "image": img_name,
        "prediction": prediction,
        "expected_building": exp_building,
        "expected_floor": exp_floor,
        "predicted_building": pred_building,
        "predicted_floor": pred_floor,
        "building_ok": building_ok,
        "floor_ok": floor_ok,
        "location_ok": location_ok
    })

# Calcular metricas finales
building_accuracy = building_correct / building_total if building_total > 0 else None
floor_accuracy = floor_correct / floor_total
location_accuracy = location_correct / location_total if location_total > 0 else None

print("="*70)
print("RESULTADOS DE LA EVALUACION")
print("="*70)
print(f"\nModelo: {MODEL_ID}")
print(f"Total imagenes evaluadas: {len(results)}")
print(f"  - Con building visible: {building_total}")
print(f"  - Solo floor: {floor_total - building_total}")

print("\n[METRICAS PRINCIPALES]")
print("-"*50)
if building_accuracy is not None:
    print(f"Building Accuracy: {building_accuracy:.2%} ({building_correct}/{building_total})")
print(f"Floor Accuracy: {floor_accuracy:.2%} ({floor_correct}/{floor_total})")
if location_accuracy is not None:
    print(f"Location Accuracy: {location_accuracy:.2%} ({location_correct}/{location_total})")
    print("  (Building AND Floor correctos - METRICA PRINCIPAL)")

print("\n[CRITERIO DE SELECCION]")
print("-"*50)
if location_accuracy is not None:
    location_pass = location_accuracy >= 0.90
    floor_pass = floor_accuracy >= 0.95
    print(f"Location Accuracy >= 0.90: {'PASS' if location_pass else 'FAIL'}")
    print(f"Floor Accuracy >= 0.95: {'PASS' if floor_pass else 'FAIL'}")
    
    if location_pass and floor_pass:
        print("\nMODELO CUMPLE CRITERIOS DE SELECCION")
    else:
        print("\nMODELO NO CUMPLE CRITERIOS DE SELECCION")
else:
    # Solo floor
    floor_pass = floor_accuracy >= 0.95
    print(f"Floor Accuracy >= 0.95: {'PASS' if floor_pass else 'FAIL'}")

print("\n" + "="*70)

In [ ]:
# ============================================================
# MOSTRAR DETALLES POR IMAGEN
# ============================================================

print("="*70)
print("DETALLES POR IMAGEN")
print("="*70)

for detail in detailed_results:
    print(f"\n[{detail['image']}]")
    print(f"  Prediccion: {detail['prediction']}")
    print(f"  Expected: Building={detail['expected_building']}, Floor={detail['expected_floor']}")
    print(f"  Detected: Building={detail['predicted_building']}, Floor={detail['predicted_floor']}")
    
    if detail['building_ok'] is not None:
        status = "OK" if detail['building_ok'] else "FAIL"
        print(f"  Building: {status}")
    
    status = "OK" if detail['floor_ok'] else "FAIL"
    print(f"  Floor: {status}")
    
    if detail['location_ok'] is not None:
        status = "OK" if detail['location_ok'] else "FAIL"
        print(f"  Location: {status}")

print("\n" + "="*70)

## 14. Resumen final

In [ ]:
# ============================================================
# RESUMEN FINAL
# ============================================================

print("="*70)
print("RESUMEN FINAL")
print("="*70)

print(f"\nModelo: {MODEL_ID}")
print(f"Total imagenes: {len(results)}")

print("\n[METRICAS FINALES]")
if building_accuracy is not None:
    print(f"  Building Accuracy: {building_accuracy:.2%}")
print(f"  Floor Accuracy: {floor_accuracy:.2%}")
if location_accuracy is not None:
    print(f"  Location Accuracy: {location_accuracy:.2%} (METRICA PRINCIPAL)")

# Tiempo promedio
avg_time = sum(r["inference_time"] for r in results) / len(results)
total_time = sum(r["inference_time"] for r in results)
print(f"\n[TIEMPOS]")
print(f"  Tiempo promedio: {avg_time:.2f}s por imagen")
print(f"  Tiempo total: {total_time:.2f}s")

print("\n[PARA LA MEMORIA]")
if location_accuracy is not None:
    if location_accuracy >= 0.90 and floor_accuracy >= 0.95:
        print("  El modelo cumple los criterios de seleccion.")
        print("  Location Accuracy >= 0.90 y Floor Accuracy >= 0.95")
    else:
        print("  El modelo no cumple todos los criterios.")
        if location_accuracy < 0.90:
            print(f"     Location Accuracy ({location_accuracy:.2%}) < 0.90")
        if floor_accuracy < 0.95:
            print(f"     Floor Accuracy ({floor_accuracy:.2%}) < 0.95")
else:
    if floor_accuracy >= 0.95:
        print("  El modelo extrae Floor correctamente.")
    else:
        print(f"  Floor Accuracy ({floor_accuracy:.2%}) < 0.95")

print("\n" + "="*70)

---

## Notas para la memoria

### Metodologia del Experimento 2:

1. **Dataset**: 15 imagenes de carteles de senalizacion del campus UFV
   - 10 imagenes con building + floor visibles
   - 5 imagenes con solo floor visible

2. **Pregunta unica**: "Read this building sign. Extract building code and floor number."

3. **Metricas evaluadas**:
   - **Building Accuracy**: Precision en extraccion de Building cuando es visible
   - **Floor Accuracy**: Precision en extraccion de Floor en todas las imagenes
   - **Location Accuracy**: Precision en extraccion de Building AND Floor cuando ambos son visibles (metrica principal)

4. **Criterio de seleccion**:
   - Location Accuracy >= 0.90 (obligatorio)
   - Floor Accuracy >= 0.95 (obligatorio)

### Observaciones:

- El modelo debe extraer correctamente Building y Floor para determinar el estado inicial
- Location Accuracy es la metrica principal porque mide la capacidad de extraer ambos campos simultaneamente
- Floor Accuracy es critica porque Floor siempre esta visible en los carteles